In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pbn_37k
from pbn_37k.ask import sigAssistant

import pandas as pd
from tqdm import tqdm
tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

brain = sigAssistant(pathCache=".cache/", llm_highend="gpt-4o",  llm_fast="gpt-4o-mini")

/home/kelu/projets/gbnexplore/.venv/lib/python3.11/site-packages/langchain/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [4]:
from langchain_community.chat_models import ChatPerplexity
from langchain.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from langchain_perplexity import ChatPerplexity


import pandas as pd
set_llm_cache(SQLiteCache(database_path=".cache/.langchain.db"))



In [5]:
perplex = ChatPerplexity(model="sonar-pro")

def askp(prompt, size="low"):

    chat = perplex
    response = chat.invoke(prompt,
                            extra_body={"web_search_options":
                                        {"search_context_size": size}}
    )
    x = response.model_dump()["content"]
    citations = response.model_dump()['additional_kwargs']["citations"]
    for n in range(len(citations)):
        x = x.replace("["+str(n+1)+"]", "(src: "+citations[n]+" )")
    return x

In [6]:
## Finding similar use cases.

prompt_template = """We are reviewing the use case of a product or service to be delivered in a GBN (green building neighorhood).
This use case takes place in the city of {Place}.

Given the following case studies description

--- 

{use_case}

--- 


You have to find me five other similar case studies that have been implemented in other similar cities, and provide as much detail as you can.

The city must be in Europe. It'd be better if we can refer to another horizon / european project for the two last case studies.

Clearly indicate the title and the number of the case studies you will find in your answer.

DONT refer to the original use case, only provide new case studies. Don't use case studies from the EU H202020 project called PROBONO.


## X. Title (replace with actual title of the case study))

* **Case study Title:** title
* **Location:** City, Country

### **Description**

description  
 
###  **Key Features:**  

- **Technology Used:** technology  
- **Sustainability Impact:** impact  
- **Implementation Challenges:** challenges  
- **Outcomes:** outcomes


### **Parallels to the use case being reviewed**  

- first parallel 
- second parallel
- etc

### **Social, Economic, Environmental Benefits:**
- first outcome 
- second outcome
- etc

"""

In [15]:
original_use_cases = pd.read_parquet("data/pbn/uc.parquet.gzip").drop_duplicates(subset=["Source"])[["Origin","Place","Type","Source","Source_Title"]].reset_index(drop=True)
NUC = len(original_use_cases)
print(NUC)
original_use_cases.head(3)

49


,Origin,Place,Type,Source,Source_Title
0,ZP_UCS--DUB-D-UC1,Dublin,UC,# Use case : DUB-D-UC1\n\n\nCity: Dublin\n\n##...,Dublin energy monitoring and management.
1,ZP_UCS--PTO-B-UC1,Porto,UC,# Use case : PTO-B-UC1\n\n\nCity: Porto\n\n## ...,Operational efficiency and sustainability impr...
2,ZP_UCS--MDC-B-UC1,Madrid,UC,# Use case : MDC-B-UC1\n\n\nCity: Madrid\n\n##...,Electricity monitoring and sustainability repo...


In [16]:
limit = NUC+1

content = []
for ix, row in original_use_cases.head(limit).iterrows():
    prompt = prompt_template.format(use_case=row["Source"], Place=row["Place"])
    response = askp(prompt, size="low")
    print(f"# Use case from {row['Origin']}\n")
    print(response)
    print("\n"+"-"*80+"\n")
    content.append({
        "Origin": row["Origin"],
        "Place": row["Place"],
        "Type": row["Type"],
        "Source": row["Source"],
        "Source_Title": row["Source_Title"],
        "Similar_use_cases": response
    })
df = pd.DataFrame(content)

NameError: name 'prompt_template' is not defined

In [9]:
prompt_detail_template = """You are reviewing a list of case studies related to green building neighborhoods (GBN) implemented in various European cities.
Have a look at the high level review of the following case study:

{case_study}



## Action for you:

Provide three pages, as complete as possible, about the details of this case study.

Use the following structure:

```
# [Official name of the project/program]

## **I. INITIATIVE OVERVIEW AND IDENTIFICATION**

### **Basic Information**
- **Initiative Title:** [Official name of the project/program]
- **Alternative Names:** [Any other names by which the initiative is known]
- **Location:** [City, Neighborhood/District, Region, Country]
- **Geographic Coordinates:** [If relevant for mapping purposes]
- **Timeline:** [Start date, key milestones, completion/ongoing status]
- **Project Scale:** [Size in area, population affected, or investment value]
- **Leading Organization(s):** [Government agencies, private developers, NGOs, or partnerships]
- **Funding Sources:** [Public, private, PPP, international aid, etc.]

### **Executive Summary**
Provide a 200-300 word overview that captures the essence of the initiative, its primary objectives, and its significance in the urban development landscape.

---

## **II. CONTEXTUAL ANALYSIS**

### **Urban Context**
- **City Profile:** Describe the host city's population, economic base, geographic characteristics, and position within regional/national urban hierarchy
- **Neighborhood/District Characteristics:** Detail the specific area where the initiative is located, including:
  - Historical development patterns
  - Current land use composition
  - Demographic profile (income levels, age distribution, ethnic/cultural composition)
  - Existing infrastructure quality and accessibility
  - Environmental conditions and constraints

### **Pre-Initiative Conditions**
- **Problems/Challenges Identified:** What specific urban issues was this initiative designed to address? (e.g., inadequate housing, traffic congestion, environmental degradation, economic stagnation, social inequality)
- **Triggering Events:** Were there specific catalysts that prompted action? (e.g., natural disasters, economic crises, policy changes, community mobilization)
- **Stakeholder Landscape:** Who were the key actors and what were their interests before the initiative began?

### **Policy and Regulatory Environment**
- **Governance Framework:** What governmental structures and authority were involved?
- **Relevant Policies:** Which urban planning policies, zoning regulations, or development strategies shaped the initiative?
- **Legal and Institutional Enablers/Barriers:** What legal frameworks facilitated or complicated the initiative?

---

## **III. INITIATIVE RATIONALE AND OBJECTIVES**

### **Strategic Vision**
- **Primary Goals:** What were the explicitly stated objectives of the initiative?
- **Vision Statement:** What aspirational future was the initiative working toward?
- **Alignment with Broader Plans:** How did this fit into citywide, regional, or national development strategies?

### **Problem-Solution Logic**
- **Theory of Change:** Explain the causal pathway from initiative activities to intended outcomes
- **Target Beneficiaries:** Who was intended to benefit, and how?
- **Innovation Elements:** What made this approach distinctive or experimental?

### **Drivers and Motivations**
- **Political Drivers:** Election cycles, political leadership priorities, public pressure
- **Economic Drivers:** Investment opportunities, economic development needs, market demands
- **Social Drivers:** Community advocacy, equity concerns, quality of life improvements
- **Environmental Drivers:** Climate adaptation, sustainability goals, resource management
- **Technological Drivers:** Availability of new technologies or smart city trends

---

## **IV. DESIGN AND IMPLEMENTATION APPROACH**

### **Project Design**
- **Conceptual Framework:** What urban design principles, planning philosophies, or development models guided the initiative?
- **Physical/Spatial Design:** Describe the built environment changes, infrastructure investments, or spatial reorganization involved
- **Scope and Phasing:** How was the project structured and sequenced over time?

### **Technologies and Innovations**
- **Technologies Deployed:** List and describe specific technologies used (e.g., smart sensors, renewable energy systems, digital platforms, construction techniques)
- **Innovation Type:** Classify innovations as technological, social, financial, governance-related, or process-oriented
- **Integration Approach:** How were technologies integrated with existing urban systems?

### **Implementation Methodology**
- **Procurement and Contracting:** How were implementers selected and engaged?
- **Stakeholder Engagement:** Describe the process for involving residents, businesses, and other stakeholders
- **Participatory Elements:** What opportunities existed for community input and co-creation?
- **Project Management:** What structures and processes were used to coordinate activities?

### **Financing Model**
- **Budget and Costs:** Total investment, cost breakdown by component
- **Funding Mix:** Proportions from different sources
- **Revenue Mechanisms:** User fees, tax increments, land value capture, or other financing innovations
- **Financial Sustainability:** Plans for long-term financial viability

---

## **V. CHALLENGES AND ADAPTIVE RESPONSES**

### **Implementation Challenges**
- **Technical Challenges:** Engineering difficulties, technology failures, infrastructure constraints
- **Financial Challenges:** Budget overruns, funding gaps, economic downturns
- **Social and Political Challenges:** Community opposition, political interference, displacement concerns, equity issues
- **Administrative Challenges:** Bureaucratic delays, coordination problems, capacity constraints
- **Environmental Challenges:** Unforeseen environmental impacts, climate events, site contamination

### **Adaptive Strategies**
- **Problem-Solving Approaches:** How were challenges addressed?
- **Design Modifications:** What changes were made to original plans?
- **Lessons Learned During Implementation:** What insights emerged that shaped the project's evolution?

---

## **VI. OUTCOMES AND IMPACTS**

### **Direct Outputs**
- **Physical Outputs:** Infrastructure created, buildings constructed, land developed
- **Service Delivery:** New or improved services provided to residents
- **Quantitative Metrics:** Numbers served, area covered, capacity created

### **Social Impacts**
- **Community Benefits:** Improvements to quality of life, health outcomes, safety, social cohesion
- **Equity Outcomes:** Distribution of benefits across different social groups, effects on vulnerable populations
- **Displacement and Gentrification:** Any unintended negative consequences for existing residents
- **Social Capital:** Changes in community networks, civic engagement, or social trust
- **Cultural Impacts:** Effects on cultural heritage, identity, or creative expression

### **Economic Impacts**
- **Job Creation:** Direct and indirect employment effects
- **Economic Activity:** Business development, investment attraction, productivity gains
- **Property Values:** Changes in real estate markets
- **Cost Savings:** Efficiencies or reduced expenses for residents or government
- **Economic Inclusion:** Effects on poverty, income inequality, or economic opportunity

### **Environmental Impacts**
- **Climate and Emissions:** Changes in greenhouse gas emissions, climate resilience
- **Resource Management:** Water, energy, material use efficiency
- **Environmental Quality:** Air quality, noise, green space, biodiversity
- **Waste and Pollution:** Improvements or deteriorations in waste management or pollution levels
- **Sustainability Metrics:** Performance against environmental standards or certifications

### **Governance and Institutional Impacts**
- **Institutional Capacity:** Strengthened government capabilities, new collaborative mechanisms
- **Policy Influence:** Inspiration for new policies or regulations
- **Transparency and Accountability:** Improvements in governance quality

### **Performance Against Objectives**
- **Success Metrics:** To what extent were stated objectives achieved?
- **Unintended Consequences:** Positive or negative outcomes not originally anticipated
- **Long-term Viability:** Evidence of sustainability and lasting impact

---

## **VII. TRANSFERABILITY AND BROADER RELEVANCE**

### **Replicability Assessment**
- **Contextual Dependencies:** Which elements were unique to the local context vs. potentially transferable?
- **Critical Success Factors:** What conditions or elements were essential to positive outcomes?
- **Scalability:** Could this approach work at different scales?

### **Comparative Analysis**
- **Similar Initiatives:** How does this compare to analogous projects in other locations?
- **Best Practices:** What elements represent exemplary practice in urban development?
- **Cautionary Lessons:** What should others avoid or approach differently?

### **Parallels to [Your Specific Use Case]**
*[This section should be customized based on the specific project or context you're comparing to]*
- **Contextual Similarities:** Shared urban conditions, challenges, or opportunities
- **Methodological Parallels:** Similar approaches to planning, implementation, or engagement
- **Technological Overlaps:** Common technologies or innovation strategies
- **Stakeholder Dynamics:** Comparable governance structures or partnership models
- **Transferable Strategies:** Specific lessons or approaches applicable to your context
- **Adaptation Considerations:** What would need to change to apply insights from this case?

---

## **VIII. CRITICAL ASSESSMENT AND REFLECTION**

### **Strengths of the Initiative**
Identify and explain the most successful or commendable aspects of the project.

### **Limitations and Weaknesses**
Honestly assess shortcomings, missed opportunities, or problematic elements.

### **Equity and Justice Considerations**
Evaluate whether the initiative advanced or hindered urban equity, environmental justice, and inclusive development.

### **Future Outlook**
- What is the long-term trajectory for this initiative?
- What ongoing challenges or opportunities exist?
- What evolution or next phases are planned or needed?

---


 **ANALYSIS INSTRUCTIONS**

When analyzing an urban development initiative using this framework:

1. **Be Comprehensive**: Address all sections thoroughly, noting when information is unavailable
2. **Be Evidence-Based**: Support claims with specific data, examples, or credible sources
3. **Be Critical**: Move beyond promotional descriptions to assess real impacts and trade-offs
4. **Be Contextual**: Always situate the initiative within its specific urban, political, and social context
5. **Be Comparative**: Draw connections to similar initiatives and broader urban development trends
6. **Be Forward-Looking**: Consider implications for future urban development practice
7. **Maintain Objectivity**: Present multiple perspectives, especially on controversial aspects
8. **Highlight Complexity**: Acknowledge that urban development outcomes are rarely purely positive or negative

The goal is to produce a nuanced, well-documented case study that serves both as a historical record and as a learning resource for urban development practitioners, policymakers, and researchers.
"""

In [10]:
df

,Origin,Place,Type,Source,Source_Title,Similar_use_cases
0,ZP_UCS--DUB-D-UC1,Dublin,UC,# Use case : DUB-D-UC1\n\n\nCity: Dublin\n\n##...,Dublin energy monitoring and management.,## 1. Living Lab BIPV – Solar Building-Integra...
1,ZP_UCS--PTO-B-UC1,Porto,UC,# Use case : PTO-B-UC1\n\n\nCity: Porto\n\n## ...,Operational efficiency and sustainability impr...,Here are **five case studies** of products or ...
2,ZP_UCS--MDC-B-UC1,Madrid,UC,# Use case : MDC-B-UC1\n\n\nCity: Madrid\n\n##...,Electricity monitoring and sustainability repo...,Here are five detailed case studies of digital...
3,ZP_UCS--BRU-B-UC2,Brussels,UC,# Use case : BRU-B-UC2\n\n\nCity: Brussels\n\n...,Energy management platform for communities.,Here are five detailed European case studies i...
4,ZP_UCS--DUB-D-UC4,Dublin,UC,# Use case : DUB-D-UC4\n\n\nCity: Dublin\n\n##...,Remote control of energy systems.,## 1. SmartEnCity – Vitoria-Gasteiz Smart Dist...
5,ZP_UCS--DUB-D-UC3,Dublin,UC,# Use case : DUB-D-UC3\n\n\nCity: Dublin\n\n##...,Energy flow management for sustainability.,Here are five case studies of energy flow moni...
6,ZP_UCS--MGP-C-UC1,Madrid,UC,# Use case : MGP-C-UC1\n\n\nCity: Madrid\n\n##...,KPI visualization and reporting tool.,Here are five detailed case studies of **green...
7,ZP_UCS--AAR-A-UC3,Aarhus,UC,# Use case : AAR-A-UC3\n\n\nCity: Aarhus\n\n##...,Human-centered building design feedback tool.,## 1. Green Hub House\n\n* **Case study Title:...
8,ZP_UCS--BRU-C-UC1,Brussels,UC,# Use case : BRU-C-UC1\n\n\nCity: Brussels\n\n...,Optimizing Brussels mobility for sustainability.,## 1. eHUBS – Smart Shared Green Mobility Hubs...
9,ZP_UCS--MDC-C-UC1,Madrid,UC,# Use case : MDC-C-UC1\n\n\nCity: Madrid\n\n##...,Energy consumption monitoring in Madrid.,## 1. Cleanwatts Living Lab – AI-Powered Renew...


In [11]:
newucs = []
responses = []
for ix, row in df.iterrows():
    nn = 0
    try:
        txt = row["Similar_use_cases"]
        if txt.startswith("\n## "):
            UCS = txt.split("\n## ")
        else:
            UCS = txt.split("\n## ")
        if len(UCS[0]) < 500:
            UCS = UCS[1:]
        for uc in UCS:
            nn += 1
            #n = uc.replace("#","").strip().split(" ")[0].strip(".")
            print(nn, row["Origin"])
            prompt = prompt_detail_template.format(case_study=uc)
            response = askp(prompt, size="high")
            responses.append(response)
            newucs.append({
                "Origin": row["Origin"],
                "Place": row["Place"],
                "Type": row["Type"],
                "Source": row["Source"],
                "Similar_use_cases": txt,
                "Similar_use_case_number": nn,
                "Similar_use_case_detail": response
            })
            print(response[:100])
            
        print("OK")
    except:
        print("error")
        pass
        

1 ZP_UCS--DUB-D-UC1
# Living Lab BIPV (Building-Integrated Photovoltaics)

---

## **I. INITIATIVE OVERVIEW AND IDENTIFI
2 ZP_UCS--DUB-D-UC1
# Ethos Engineering Living Lab Digital Twin

## **I. INITIATIVE OVERVIEW AND IDENTIFICATION**

### *
3 ZP_UCS--DUB-D-UC1
# Integrated Energy Lab (IE Lab), UCD Energy Institute

---

## **I. INITIATIVE OVERVIEW AND IDENTIF
4 ZP_UCS--DUB-D-UC1
# Energy Living Lab (Provincia di Sassari, Italy)

---

## **I. INITIATIVE OVERVIEW AND IDENTIFICATI
5 ZP_UCS--DUB-D-UC1
# Virtual Heating Plant Gleisdorf

---

## **I. INITIATIVE OVERVIEW AND IDENTIFICATION**

### **Basi
OK
1 ZP_UCS--PTO-B-UC1
# Agra do Amial Renewable Energy Community

---

## **I. INITIATIVE OVERVIEW AND IDENTIFICATION**

#
2 ZP_UCS--PTO-B-UC1
# FUN Porto (Urban Forest and BioSpots Network)

---

## **I. INITIATIVE OVERVIEW AND IDENTIFICATION
3 ZP_UCS--PTO-B-UC1
# LIFE-myBUILDINGisGREEN: Falcão Primary School Green Roof Demonstration

## **I. INITIATIVE OVERVIE
4 ZP_UCS--PTO-B-UC1
# Porto d

In [12]:
d = pd.DataFrame(newucs)[["Origin","Similar_use_case_number","Similar_use_case_detail"]]
d[87:90]

,Origin,Similar_use_case_number,Similar_use_case_detail
87,ZP_UCS--DUB-B-UC3,5,# RESPONSE Dijon – Positive Energy Block\n\n--...
88,ZP_UCS--MGP-A-UC1,1,# Triple – Eco-Carbon Neutral Co-Working and E...
89,ZP_UCS--MGP-B-UC3,1,# Madrid Nuevo Norte Urban Geothermal Power Ne...


# Scoring

In [13]:
len(responses)

238

In [14]:
analyses = []
for response in responses:
    print(response[:20])
    analysis = brain.analyseText( txt=response,
            TypeOfItem = "Activity",
            Source="open_web_review",
            Place="TBC",
            MIN=7,
            Reviewed=False,
            MODEL="gpt-4o-mini")
    analyses.append(analysis)
analyses = pd.concat(analyses)

# Living Lab BIPV (B
Already done
# Ethos Engineering 
Already done
# Integrated Energy 
Already done
# Energy Living Lab 
Already done
# Virtual Heating Pl
Already done
# Agra do Amial Rene
Already done
# FUN Porto (Urban F
Already done
# LIFE-myBUILDINGisG
Already done
# Porto di Mare Eco-
Already done
# Sponge Parks and S
Already done
# Casa Sophia – Posi
Already done
# Orcasitas Communit
Already done
# HABITA-RES Urban N
Already done
# mySMARTLife – Smar
Already done
# Sharing Cities – S
Already done
# Tivoli GreenCity –
Already done
# GreenBizz Energy C
Already done
# Be.SHARE – Brussel
Already done
# RESPONSE – integRa
Already done
# Sharing Cities – S
Already done
# SmartEnCity – Vito
Already done
# Dockline A3 BER Re
Already done
# EnergyVille Smart 
Already done
# mySMARTLife Urban 
Already done
# IRIS Smart Cities 
Already done
# Amsterdam Smart En
Already done
# Aspern Smart City 
Already done
# Lyon Confluence Sm
Already done
# REMOURBAN – Nottin
Already done
# mySMARTLife 

# Saving

In [15]:
d = d[["Origin","Similar_use_case_number","Similar_use_case_detail"]].drop_duplicates()
d.columns = ["ZP_UC","Similar_use_case_number","Similar_use_case_detail"]
d

,ZP_UC,Similar_use_case_number,Similar_use_case_detail
0,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...
1,ZP_UCS--DUB-D-UC1,2,# Ethos Engineering Living Lab Digital Twin\n\...
2,ZP_UCS--DUB-D-UC1,3,"# Integrated Energy Lab (IE Lab), UCD Energy I..."
3,ZP_UCS--DUB-D-UC1,4,"# Energy Living Lab (Provincia di Sassari, Ita..."
4,ZP_UCS--DUB-D-UC1,5,# Virtual Heating Plant Gleisdorf\n\n---\n\n##...
...,...,...,...
233,ZP_UCS--DUB-C-UC1,1,# Food Smart Schools Initiative\n\n## **I. INI...
234,ZP_UCS--DUB-C-UC1,2,# Smart Waste Reduction in Canteens (Gothenbur...
235,ZP_UCS--DUB-C-UC1,3,# Eco-Canteen Food Waste Management Initiative...
236,ZP_UCS--DUB-C-UC1,4,"# Green Canteen Digital Twin – Helsinki, Finla..."


In [16]:
memory = analyses.merge(d, left_on=["Source"], right_on=["Similar_use_case_detail"])
memory["id"] = memory["ZP_UC"] + "_" + memory["Similar_use_case_number"].astype(str)
memory

,FromProbono,Origin,Place,Type,Source,Justification,Purpose,Issue,Scale,Score,Justification_Short,Source_Title,Reviewed,model,timestamp,reviewsrc,ZP_UC,Similar_use_case_number,Similar_use_case_detail,id
0,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The Living Lab BIPV represents a pioneering in...,Attractiveness,"Innovation, creativity and research",Building,5,Innovative building-integrated photovoltaics m...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
1,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The initiative focuses on reducing the environ...,Preservation and improvement of environment,Biodiversity and ecosystem services,Building,4,Sustainable buildings with renewable energy.,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
2,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,"By integrating renewable energy solutions, the...",Resilience,Health and care in the community,Neighbourhood,4,Renewable energy boosts community resilience.,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
3,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The Living Lab BIPV emphasizes efficient resou...,Responsible resource use,Economy and sustainable production and consump...,Building,5,Energy-generating building materials promote s...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
4,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,Though primarily focused on technological inno...,Social cohesion,"Living together, interdependence and mutuality",Neighbourhood,3,Living Lab promotes collaboration and sustaina...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2272,False,open_web_review,TBC,Activity,"# Circular Canteen Management – Amsterdam, Net...",By reducing food waste and improving the quali...,Well-being,Health and care in the community,Neighbourhood,4,Improving food quality enhances employee welln...,Circular canteen waste reduction initiative.,False,gpt-4o-mini,"11/22/2025, 12:07:55",N/A,ZP_UCS--DUB-C-UC1,5,"# Circular Canteen Management – Amsterdam, Net...",ZP_UCS--DUB-C-UC1_5
2273,False,open_web_review,TBC,Activity,"# Circular Canteen Management – Amsterdam, Net...","The circular canteen initiative, by providing ...",Attractiveness,Mobility,Neighbourhood,3,Local food services enhance staff mobility.,Circular canteen waste reduction initiative.,False,gpt-4o-mini,"11/22/2025, 12:07:55",N/A,ZP_UCS--DUB-C-UC1,5,"# Circular Canteen Management – Amsterdam, Net...",ZP_UCS--DUB-C-UC1_5
2274,False,open_web_review,TBC,Activity,"# Circular Canteen Management – Amsterdam, Net...","By focusing on local sourcing of food, the ini...",Preservation and improvement of environment,Biodiversity and ecosystem services,Neighbourhood,3,Supports biodiversity through local sourcing.,Circular canteen waste reduction initiative.,False,gpt-4o-mini,"11/22/2025, 12:07:55",N/A,ZP_UCS--DUB-C-UC1,5,"# Circular Canteen Management – Amsterdam, Net...",ZP_UCS--DUB-C-UC1_5
2275,False,open_web_review,TBC,Activity,"# Circular Canteen Management – Amsterdam, Net...",The initiative emphasizes stakeholder engageme...,Resilience,"Governance, 

In [17]:
memory.to_parquet("output/activities_open_web.parquet.gzip", compression="gzip")

# Create pages

In [6]:
import pandas as pd

In [7]:
memory = pd.read_parquet("output/activities_open_web.parquet.gzip")
memory.head(2)

,FromProbono,Origin,Place,Type,Source,Justification,Purpose,Issue,Scale,Score,Justification_Short,Source_Title,Reviewed,model,timestamp,reviewsrc,ZP_UC,Similar_use_case_number,Similar_use_case_detail,id
0,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The Living Lab BIPV represents a pioneering in...,Attractiveness,"Innovation, creativity and research",Building,5,Innovative building-integrated photovoltaics m...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1
1,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The initiative focuses on reducing the environ...,Preservation and improvement of environment,Biodiversity and ecosystem services,Building,4,Sustainable buildings with renewable energy.,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1


In [8]:
HEADER = """---
layout: default
title: "TITLE"
parent: PARENT
has_children: true
nav_order: NAVORDER
---\n\n"""

In [9]:

def convert_links_to_footnotes(text):
    """
    Convert (src: LINK ) patterns to markdown footnotes and add footnote list at the end.
    
    Args:
        text (str): Input text containing (src: LINK ) patterns
        
    Returns:
        str: Text with footnotes replaced and footnote list appended
    """
    # Pattern to match (src: LINK ) - captures the link part
    pattern = r'\(src:\s*([^)]+)\s*\)'
    
    # Find all matches
    matches = re.findall(pattern, text)
    
    if not matches:
        return text
    
    # Remove duplicates while preserving order
    unique_links = []
    seen = set()
    for link in matches:
        link = link.strip()
        if link not in seen:
            unique_links.append(link)
            seen.add(link)
    
    # Create a mapping from link to footnote number
    link_to_footnote = {link: i + 1 for i, link in enumerate(unique_links)}
    
    # Replace each occurrence with the appropriate footnote
    def replace_match(match):
        link = match.group(1).strip()
        footnote_num = link_to_footnote[link]
        return f'[^{footnote_num}]'
    
    # Replace all occurrences
    result_text = re.sub(pattern, replace_match, text)
    
    # Add footnote list at the end
    footnote_list = '\n\n'
    for link, num in link_to_footnote.items():
        footnote_list += f'[^{num}]: {link}\n'
    
    return result_text + footnote_list.rstrip()


In [10]:
import re

## Individual use cases

In [11]:
c = 0
memory["col1"] = memory["Purpose"] + " x "+ memory["Issue"]
for id in memory["id"].unique():
    c += 1
    txt = memory[memory["id"]==id]["Similar_use_case_detail"].values[0]
    UCID = memory[memory["id"]==id]["ZP_UC"].values[0].replace("-","").replace("_","")
    title = txt.split("\n")[0].strip("#").strip().replace(":"," ")
    table = memory[memory["id"]==id][["col1","Justification"]]
    print(id, UCID, title)
    MD = HEADER.replace("TITLE",title).replace("NAVORDER",str(c+4)).replace("PARENT",UCID)
    MD += "\n# "+title+"\n\n"+"# Evaluation\n\n"+table.to_markdown(index=False)+"\n\n# Executive summary"+convert_links_to_footnotes(txt).replace("# ", "## ")

    with open("docs/activities_"+id+".md", "w") as f:
        f.write(MD)

ZP_UCS--DUB-D-UC1_1 ZPUCSDUBDUC1 Living Lab BIPV (Building-Integrated Photovoltaics)
ZP_UCS--DUB-D-UC1_2 ZPUCSDUBDUC1 Ethos Engineering Living Lab Digital Twin
ZP_UCS--DUB-D-UC1_3 ZPUCSDUBDUC1 Integrated Energy Lab (IE Lab), UCD Energy Institute
ZP_UCS--DUB-D-UC1_4 ZPUCSDUBDUC1 Energy Living Lab (Provincia di Sassari, Italy)
ZP_UCS--DUB-D-UC1_5 ZPUCSDUBDUC1 Virtual Heating Plant Gleisdorf
ZP_UCS--PTO-B-UC1_1 ZPUCSPTOBUC1 Agra do Amial Renewable Energy Community
ZP_UCS--PTO-B-UC1_2 ZPUCSPTOBUC1 FUN Porto (Urban Forest and BioSpots Network)
ZP_UCS--PTO-B-UC1_3 ZPUCSPTOBUC1 LIFE-myBUILDINGisGREEN  Falcão Primary School Green Roof Demonstration
ZP_UCS--PTO-B-UC1_4 ZPUCSPTOBUC1 Porto di Mare Eco-District
ZP_UCS--PTO-B-UC1_5 ZPUCSPTOBUC1 Sponge Parks and Social Housing Energy Communities (EXHAUSTION Horizon Project)
ZP_UCS--MDC-B-UC1_1 ZPUCSMDCBUC1 Casa Sophia – Positive Energy Smart Home
ZP_UCS--MDC-B-UC1_2 ZPUCSMDCBUC1 Orcasitas Community Energy Renovation
ZP_UCS--MDC-B-UC1_3 ZPUCSMDCBUC1 

## Original Use cases

In [12]:
memory.head(2)

,FromProbono,Origin,Place,Type,Source,Justification,Purpose,Issue,Scale,Score,...,Source_Title,Reviewed,model,timestamp,reviewsrc,ZP_UC,Similar_use_case_number,Similar_use_case_detail,id,col1
0,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The Living Lab BIPV represents a pioneering in...,Attractiveness,"Innovation, creativity and research",Building,5,...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1,"Attractiveness x Innovation, creativity and re..."
1,False,open_web_review,TBC,Activity,# Living Lab BIPV (Building-Integrated Photovo...,The initiative focuses on reducing the environ...,Preservation and improvement of environment,Biodiversity and ecosystem services,Building,4,...,Building-integrated photovoltaics research ini...,False,gpt-4o-mini,"11/22/2025, 09:42:44",N/A,ZP_UCS--DUB-D-UC1,1,# Living Lab BIPV (Building-Integrated Photovo...,ZP_UCS--DUB-D-UC1_1,Preservation and improvement of environment x ...


In [ ]:
c = 5
for id in memory["ZP_UC"].unique():
    c += 1
    txt = df[df["Origin"]==id]["Source"].values[0]
    title = df[df["Origin"]==id]["Source_Title"].values[0].replace(":"," ")


    MD = HEADER.replace("TITLE",id.replace("-","").replace("_","")).replace("NAVORDER",str(c+4)).replace("\nparent: PARENT","")
    MD += "\n# "+str(id)+" : " +title+"\n\n"

    MD += "# Related use cases found online\n\n"
    for ix, row in memory[memory["ZP_UC"] == id].drop_duplicates(subset= ["Similar_use_case_detail"]).iterrows():
        titl = row["Similar_use_case_detail"].split("\n")[0].strip("#").strip()
        
        MD += "* ["+titl+"](activities_"+row["id"]+".md)\n"

    MD += "\n\n# Original text\n\n"+convert_links_to_footnotes(txt).replace("# ", "## ")

    with open("docs/uc_"+id+".md", "w") as f:
        f.write(MD)

NameError: name 'df' is not defined